# 05 - Evaluation
Run full evaluation across all samples with all three methods.
Generates Tables 2, 3, 4, 5 from the paper.

In [1]:
import sys, os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
import numpy as np
import pandas as pd

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import load_config, ensure_dirs
from src.data.dataset_index import DatasetIndex
from src.experiments.run_batch import run_batch_experiment
from src.experiments.summarize import summarize_results

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
cfg = load_config(os.path.join(PROJECT_ROOT, 'configs', 'chair_leg.yaml'))
ensure_dirs(cfg)
index = DatasetIndex(cfg['paths']['raw_data_dir'])
print(f"Running evaluation on {len(index)} samples...")

Running evaluation on 100 samples...


In [3]:
# Run batch experiment
df = run_batch_experiment(
    data_dir=cfg['paths']['raw_data_dir'],
    output_dir=cfg['paths']['output_dir'],
    margin=cfg['repair']['margin'],
    proximity_threshold=cfg['repair']['proximity_threshold'],
    save_meshes=True,
)

print(f"\nResults: {len(df)} samples evaluated")
df.head()

2026-04-14 21:28:05 [INFO] BatchExperiment: Running on 100 samples (SOTA=True, distance=True, mlp=no)
Running experiments: 100%|█████████████████████████████████████████████████████████| 100/100 [1:50:12<00:00, 66.13s/it]
2026-04-14 23:18:18 [INFO] BatchExperiment: Done: 84 success, 16 failed



Results: 84 samples evaluated


,sample_id,n_boundary_loops,n_target_loops_rpa,center_fan/target_loop_length_before,center_fan/target_loop_length_after,center_fan/improvement,center_fan/closure_ratio,center_fan/n_new_vertices,center_fan/n_new_faces,center_fan/avg_new_face_quality,...,advancing_front_rpa/std_new_face_quality,advancing_front_rpa/n_faces_inside,advancing_front_rpa/n_faces_outside,advancing_front_rpa/locality_ratio,advancing_front_rpa/chamfer_distance,advancing_front_rpa/hausdorff_distance,advancing_front_rpa/dev_mean,advancing_front_rpa/dev_max,advancing_front_rpa/dev_median,advancing_front_rpa/dev_std
0,2882,128,40,13.685422,0.0,13.685422,1.0,40.0,1488.0,0.289861,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,40584,24,18,4.972269,0.0,4.972269,1.0,18.0,876.0,0.254221,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,42201,32,32,19.469631,0.0,19.469631,1.0,32.0,1607.0,0.198347,...,0.122496,1543.0,0.0,1.0,0.106983,0.673196,0.010182,0.031442,0.009960,0.004834
3,41898,32,24,2.674532,0.0,2.674532,1.0,24.0,534.0,0.530598,...,0.199315,486.0,0.0,1.0,0.027688,0.339748,0.011349,0.049433,0.010519,0.006147
4,39573,163,107,68.148484,0.0,68.148484,1.0,107.0,3826.0,0.266503,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# Generate summary tables
tables = summarize_results(df, cfg['paths']['output_dir'])

print("=" * 80)
print("Table 2: Overall Quantitative Comparison")
print("=" * 80)
print(tables['table2_quantitative'].to_string(index=False, float_format='%.4f'))

Table 2: Overall Quantitative Comparison
          Method  Residual ↓  Improvement ↑  New Vtx ↓  New Faces ↓  Avg Quality ↑  Min Quality ↑
Center-fan + RPA      0.0000        39.9517    43.7500    1761.3690         0.2671         0.0725
    Planar + RPA      1.9384        38.0133     0.0000    1656.9405         0.4341         0.1333
 MLP Patch + RPA      0.0000        39.9517    43.7500    1761.3690         0.2671         0.0725
Planar + LH-only     38.2569         1.6948     0.0000      96.1905         0.4122         0.2016


In [5]:
print("=" * 80)
print("Table 3: Locality Analysis")
print("=" * 80)
print(tables['table3_locality'].to_string(index=False, float_format='%.4f'))

Table 3: Locality Analysis
          Method  Inside ↑  Outside ↓  Locality ↑
Center-fan + RPA 1065.2976   696.0714      0.6656
    Planar + RPA 1013.6786   643.2619      0.6710
 MLP Patch + RPA 1065.2976   696.0714      0.6656
Planar + LH-only   55.2857    40.9048      0.5717
Trimesh fill-all    8.4524    58.2500      0.0657
 Adv-front + RPA  920.1250   505.4792      0.7174
   Poisson recon    0.0000     0.0000      0.0000


In [6]:
print("=" * 80)
print("Table 4: Failure Cases of Largest-Hole-Only")
print("=" * 80)
t4 = tables['table4_failures']
if len(t4) > 0:
    print(t4.to_string(index=False, float_format='%.4f'))
else:
    print("No failure cases found (all samples have same selection).")

Table 4: Failure Cases of Largest-Hole-Only
   ID  RP Residual  LH Residual  LH Outside  RP Improv  LH Improv      Gap
35123      25.2602     292.8683      4.0000   269.7744     2.1663 267.6081
36267      21.3256     274.1738      4.0000   255.0144     2.1663 252.8481
  328      24.7460     229.2320      4.0000   206.6523     2.1663 204.4860
48065       0.9975     189.0620      8.0000   191.6030     3.5385 188.0644
37101       6.9858     181.0308      2.0000   177.8095     3.7644 174.0450
41573       0.3486     123.7866    122.0000   127.9022     4.4641 123.4380
 2955       0.0000      96.4979      3.0000    99.4730     2.9751  96.4979
41483       3.8411      77.6902    293.0000    78.6319     4.7828  73.8491
39573       0.0000      64.7920     62.0000    68.1485     3.3565  64.7920
43875       0.5168      62.1780    189.0000    64.8364     3.1752  61.6611


In [7]:
print("=" * 80)
print("Table 5: Target Selection Comparison (same planar geometry)")
print("=" * 80)
print(tables['table5_target_selection'].to_string(index=False, float_format='%.4f'))

Table 5: Target Selection Comparison (same planar geometry)
            Method  Residual ↓  Median ↓  Improvement ↑  Outside ↓  Locality ↑  Quality ↑
 Largest-hole-only     38.2569   18.6599         1.6948    40.9048      0.5717     0.4122
Removed-part-aware      1.9384    0.0000        38.0133   643.2619      0.6710     0.4341


In [8]:
# Save all tables as LaTeX
tables_dir = os.path.join(cfg['paths']['output_dir'], 'tables')
for name, tbl in tables.items():
    latex_path = os.path.join(tables_dir, f"{name}.tex")
    tbl.to_latex(latex_path, index=False, float_format='%.4f')
    print(f"Saved {latex_path}")

Saved D:\MyJupyter\Works\3DPART_v2\outputs\tables\table2_quantitative.tex
Saved D:\MyJupyter\Works\3DPART_v2\outputs\tables\table3_locality.tex
Saved D:\MyJupyter\Works\3DPART_v2\outputs\tables\table4_failures.tex
Saved D:\MyJupyter\Works\3DPART_v2\outputs\tables\table5_target_selection.tex
Saved D:\MyJupyter\Works\3DPART_v2\outputs\tables\table6_sota_comparison.tex
Saved D:\MyJupyter\Works\3DPART_v2\outputs\tables\table7_distance.tex
Saved D:\MyJupyter\Works\3DPART_v2\outputs\tables\table8_mlp_comparison.tex
